In [ ]:
import os, sys
import time
from pathlib import Path
from collections import Counter
from concurrent.futures import ProcessPoolExecutor

import numpy as np
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from adjustText import adjust_text

In [ ]:
# Load custom library and config
base_path = Path('../')
sys.path.append(base_path)
from helpers import APA_tools

with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
CONTRAST = 'DxPC1'
de_path_prefix = '/sc/arion/projects/CommonMind/yeon/p/APA/run_dreamlet_SubID/'

de_path_pas = de_path_prefix + 'FULL-class-PAS/2_run_dream/NUM-DxPC1_z/topTable.tsv.gz'
de_path_gene = de_path_prefix + 'FULL-class-Gene/2_run_dream/NUM-DxPC1_z/topTable.tsv.gz'

du_path = '/sc/arion/projects/CommonMind/yeon/p/APA/run_crumblr_SubID/pergene_3utr_down_pas_ratio/' +\
          'manual_FULL-class_AD_DxPC1_3utr_down_pas_ratio/3UTR-down-PAS_usage_crumblr_FULL_class_DxPC1_z.tsv.gz'

print(time.ctime(os.path.getmtime(de_path_pas)))
print(time.ctime(os.path.getmtime(de_path_gene)))
print(time.ctime(os.path.getmtime(du_path)))

In [ ]:
# Read celltypes
from Bio import Phylo

treepath = f"/sc/arion/projects/psychAD/NPS-AD/freeze2_proc/231109_PsychAD_capstone_F2/tree_class_um.nwk"
tree = Phylo.read(treepath, 'newick')
celltypes = [t.name for t in tree.get_terminals()][::-1]
celltypes

# Read DEG and DEPAS results

In [ ]:
df_tt_gene = pd.read_table(de_path_gene)
df_tt_gene

In [ ]:
FDR_CUTOFF = 0.05

df_tt_gene['DEG'] = np.select(
    [
        ((df_tt_gene['adj.P.Val']<FDR_CUTOFF) & (df_tt_gene['logFC']<0.0)),
        ((df_tt_gene['adj.P.Val']<FDR_CUTOFF) & (df_tt_gene['logFC']>0.0)),
    ],
    ['Down', 'Up'],
    default='Not_DEG'
)

In [ ]:
df_tt_pas = pd.read_table(de_path_pas)
df_tt_pas

# Load gene - PAS info

In [ ]:
tsv_path = cfg['root'] / 'ref/PAS_merged_evaluated_with_name_ageXclass.tsv'
df_pas_gene = pd.read_table(tsv_path, index_col=0, dtype={1: 'category'})
df_pas_gene

In [ ]:
pas_to_allgenes = dict(zip(df_pas_gene['PAS_name'], df_pas_gene['mapped_genes']))
pas_to_repr_genenames = dict(zip(df_pas_gene['PAS_name'], df_pas_gene['gene_name']))
repr_gene_to_pas_counts = df_pas_gene.groupby('gene_name').size()
pas_to_visualname = dict(zip(df_pas_gene['PAS_name'], df_pas_gene['visual_name']))
pas_to_index = dict(zip(df_pas_gene['PAS_name'], df_pas_gene['PAS_index_from5p']))

In [ ]:
df_tt_pas['visual_name'] = df_tt_pas['ID'].map(pas_to_visualname)
df_tt_pas['mapped_genes'] = df_tt_pas['ID'].map(pas_to_allgenes)
df_tt_pas['PAS_inde_from_5p'] = df_tt_pas['ID'].map(pas_to_index)

# Make PAS weight

In [ ]:
keep_regions = '3UTR 3primeExtended'.split()

print(set(df_pas_gene['pas_region']))
df_pas_gene = df_pas_gene[df_pas_gene['pas_region'].isin(keep_regions)].copy()
print(len(df_pas_gene))
print(set(df_pas_gene['pas_region']))

In [ ]:
df_gene_weights = APA_tools.make_pas_weights(df_pas_gene)
df_gene_weights

# Read crumblr results

In [ ]:
df_tt_dupas = pd.read_table(du_path)
df_tt_dupas

In [ ]:
# There should be some duplicated pas
print(df_tt_dupas.duplicated(subset=['assay', 'PAS_name']).sum())
# There should be no duplicated pas here
print(df_tt_dupas.duplicated(subset=['assay', 'PAS_name', 'gene_name']).sum())

Counter(df_tt_dupas.SigGroup)

# Merge and calculate WUI

In [ ]:
Counter(df_tt_dupas.SigGroup)

In [ ]:
signi_str = 'q < 0.05 & usage > 5%'
low_p_str = 'p < 0.05'
notsigni_str = 'Not Significant'

def get_sig_str(se):
    if (se==signi_str).any():
        return signi_str
    elif (se==low_p_str).any():
        return low_p_str
    else:
        return notsigni_str

wui_tables = []

def process_celltype(ct):
    df_tt_dupas_ct = df_tt_dupas[df_tt_dupas['assay']==ct]
    
    df_weight_usg = pd.merge(df_gene_weights[['PAS_name', 'pas_weight', 'gene_name']], 
                             df_tt_dupas_ct[['PAS_name', 'gene_name', 'AveUsage', 'TestUsage', 'OtherUsage', 'assay', 'P.Value', 'adj.P.Val', 'SigGroup']],
                             left_on=['PAS_name', 'gene_name'], right_on=['PAS_name', 'gene_name'], how='left')
    
    df_weight_usg['weight_TestUsage'] = df_weight_usg['pas_weight'] * df_weight_usg['TestUsage']
    df_weight_usg['weight_OtherUsage'] = df_weight_usg['pas_weight'] * df_weight_usg['OtherUsage']
    df_weight_usg['weight_AveUsage'] = df_weight_usg['pas_weight'] * df_weight_usg['AveUsage']
    gb_weight_usg = df_weight_usg.groupby('gene_name')
    
    df_wui_ct = pd.DataFrame({
        'TestWUI': gb_weight_usg['weight_TestUsage'].sum(),
        'CtrlWUI': gb_weight_usg['weight_OtherUsage'].sum(),
        'AveWUI': gb_weight_usg['weight_AveUsage'].sum(),
        'min.P.Value': gb_weight_usg['P.Value'].min(),
        'min.adj.P.Val': gb_weight_usg['adj.P.Val'].min(),
        'SigGroup': gb_weight_usg['SigGroup'].apply(get_sig_str)
    })
    df_wui_ct['assay'] = ct
    se_not_all_NA = ~df_weight_usg.groupby('gene_name')['assay'].apply(lambda se: se.isnull().all())
    print('processed:', ct)
    return df_wui_ct[se_not_all_NA].reset_index()

# Run in parallel
with ProcessPoolExecutor(max_workers=len(celltypes)) as executor:
    wui_tables = list(executor.map(process_celltype, celltypes))
    
df_wui = pd.concat(wui_tables)
df_wui.reset_index(inplace=True)
del df_wui['index']
df_wui['dWUI'] = df_wui.TestWUI - df_wui.CtrlWUI
df_wui['anySigni'] = df_wui['SigGroup']==signi_str

# Mark significance & draw dWUI

In [ ]:
df_wui['Shorter'] = df_wui.dWUI < 0
df_wui['Longer'] = df_wui.dWUI > 0

conditions = [
    df_wui['Shorter'] & df_wui['anySigni'],
    df_wui['Longer'] & df_wui['anySigni']
]
choices = [
    'Shorter & signi',
    'Longer & signi'
]
df_wui['sig_direction'] = np.select(conditions, choices, default='Other')

se_sig_counts = df_wui.groupby('assay')['sig_direction'].apply(lambda x: Counter(x)).fillna(0).astype(int)
se_sig_counts
#se_sig_counts.loc['Astro', 'Longer & signi']

## Volcano

In [ ]:
## Calculate -log10(p-value)
df_wui['neglog10_p'] = -np.log10(df_wui['min.P.Value'])

g = sns.relplot(
    data=df_wui,
    x='dWUI',
    y='neglog10_p',
    hue='sig_direction',
    col='assay',
    kind='scatter',
    palette={'Shorter & signi': 'red', 'Longer & signi': 'blue', 'Other': 'gray'},
    facet_kws={'sharey': False, 'sharex': True},
    col_wrap=4,  # Adjust based on number of assays
    height=2.5, 
    aspect=1,
    rasterized=True,
    s=10, alpha=0.5
)

g.set_axis_labels("ΔWUI", r"$-log_{10}$(min(p-value))")
g.set_titles("{col_name}")
g.set(ylim=(0, None))


df_label = df_wui.sort_values('neglog10_p', ascending=False).groupby(['assay', 'Longer']).head(3)
for ax in g.axes.flat:
    ct = ax.get_title()
    ax.text(0.99, 0.01, f'N={se_sig_counts.loc[ct, "Longer & signi"]:d}', color='blue', ha='right', va='bottom', transform=ax.transAxes, fontsize=9)
    ax.text(0.01, 0.01, f'N={se_sig_counts.loc[ct, "Shorter & signi"]:d}', color='red', ha='left', va='bottom', transform=ax.transAxes, fontsize=9)
    
    df_sub = df_label[df_label['assay']==ct]
    texts = []
    for _, row in df_sub.iterrows():
        texts.append(ax.text(row['dWUI'], row['neglog10_p'], row['gene_name'], clip_on=True, fontsize=8))
    adjust_text(texts, ax=ax, 
                avoid_self=True,
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

g.figure.suptitle(Path(du_path).name.replace('.tsv.gz', ''), y=1.015)

plt.savefig('volcano_dWUI_' + Path(du_path).name.replace('tsv.gz', 'pdf'), bbox_inches='tight', dpi=300)
plt.show()

## Violin

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 3))

cells_to_draw = 'IN EN Oligo OPC Astro Mural Immune Endo'.split()

sns.violinplot(
    data=df_wui[df_wui.assay.isin(cells_to_draw)],
    x='assay',
    y='dWUI',
    hue='anySigni',
    palette={True: '#2ca25f', False: 'lightgray'},
    ax=ax,
    density_norm='width',
    order=cells_to_draw, 
    inner_kws=dict(box_width=3, zorder=20, color='k', marker='o'),
    cut=0
)

se_min = df_wui[df_wui.assay.isin(cells_to_draw)].groupby(['assay', 'anySigni'])['dWUI'].min()
se_max = df_wui[df_wui.assay.isin(cells_to_draw)].groupby(['assay', 'anySigni'])['dWUI'].max()
se_counts = df_wui[df_wui.assay.isin(cells_to_draw)].groupby(['assay', 'anySigni'])['gene_name'].count()

offset = 0.001
xoffset = 0.2
for i, ct in enumerate(cells_to_draw):
    n_nonsig = se_counts.get((ct, False), 0)
    ypos_nonsig = se_max.get((ct, False)) + offset
    #ax.text(i-xoffset, ypos_nonsig, f'{n_nonsig}', ha='center', va='bottom', fontsize=10, color='#444444',
    #        bbox=dict(facecolor='white', edgecolor='none', pad=0.01, alpha=0.5))
    n_sig = se_counts.get((ct, True), 0)
    if n_sig > 0:
        ypos_sig = se_min.get((ct, True)) - offset
        ax.text(i+xoffset, ypos_sig, f'{n_sig}', ha='center', va='top', fontsize=10, color='darkgreen', 
                bbox=dict(facecolor='white', edgecolor='none', pad=0.01, alpha=0.5))

ax.grid(True, axis='y', zorder=-10)
ax.axhline(0, ls='--', color='k', zorder=5)

ax.set_ylim([-0.04, 0.04])
margin = 0.4
ax.set_xlim([0-xoffset-margin, len(cells_to_draw)-1+xoffset+margin])
ax.set_ylabel("ΔWUI,\nAD − Ctrl (DxPC1)")
ax.legend(loc=(1.01, 0.0))
ax.set_xticks(np.arange(len(cells_to_draw)))
ax.set_xticklabels(cells_to_draw, rotation=30, rotation_mode='anchor', ha='right', va='top')
#ax.set_xticklabels(xlabels, rotation=30, rotation_mode='anchor', ha='right', va='top')

ax.set_title(Path(du_path).name.replace('.tsv.gz', ''))

plt.savefig('violin_dWUI_' + Path(du_path).name.replace('tsv.gz', 'pdf'), bbox_inches='tight', dpi=300)
plt.show()

# Correlation between dWUI and DEG

In [ ]:
df_merged = pd.merge(df_wui, df_tt_gene, left_on=['assay', 'gene_name'], right_on=['assay', 'ID'], how='outer')
df_merged = df_merged.sort_values(by='assay', key=lambda x: x.map(lambda y: celltypes.index(y)))

In [ ]:
df_wui_geneexp = df_merged.dropna()
# Calculate -log10(p-value)

ct_to_corrs = {}
for ct in celltypes:
    df_cut = df_wui_geneexp[df_wui_geneexp['assay']==ct]
    corr, p = spearmanr(df_cut['logFC'], df_cut['dWUI'])
    ct_to_corrs[ct] = corr

g = sns.relplot(
    data=df_wui_geneexp,
    x='dWUI',
    y='logFC',
    col='assay',
    kind='scatter',
    facet_kws={'sharey': False, 'sharex': False},
    hue='sig_direction',
    palette={'Shorter & signi': 'red', 'Longer & signi': 'blue', 'Other': 'gray'},
    col_wrap=4,  # Adjust based on number of assays
    height=2.5, 
    aspect=1,
    rasterized=True,
    s=10, alpha=0.3
)

g.set_axis_labels("ΔWUI", 'logFC')
g.set_titles("{col_name}")

for ax in g.axes.flat:
    ct = ax.get_title()
    ax.text(0.01, 0.99, f'r = {ct_to_corrs[ct]:.2f}', ha='left', va='top', transform=ax.transAxes, fontsize=10)

plt.show()

In [ ]:
df_wui_geneexp = df_merged.dropna()
df_wui_geneexp = df_wui_geneexp[df_wui_geneexp.anySigni]
# Calculate -log10(p-value)

ct_to_corrs = {}
for ct in celltypes:
    df_cut = df_wui_geneexp[df_wui_geneexp['assay']==ct]
    corr, p = spearmanr(df_cut['logFC'], df_cut['dWUI'])
    ct_to_corrs[ct] = corr

g = sns.relplot(
    data=df_wui_geneexp,
    x='dWUI',
    y='logFC',
    col='assay',
    kind='scatter',
    facet_kws={'sharey': False, 'sharex': False},
    hue='sig_direction',
    palette={'Shorter & signi': 'red', 'Longer & signi': 'blue', 'Other': 'gray'},
    col_wrap=4,  # Adjust based on number of assays
    height=2.5, 
    aspect=1,
    rasterized=True,
    zorder=10,
    s=10, alpha=0.3
)

g.set_axis_labels("ΔWUI", 'logFC')
g.set_titles("{col_name}")

for ax in g.axes.flat:
    ct = ax.get_title()
    ax.grid(zorder=-1)
    ax.text(0.5, 0.99, f'r = {ct_to_corrs[ct]:.2f}', ha='center', va='top', transform=ax.transAxes, fontsize=10)


plt.show()

In [ ]:
df_wui_geneexp = df_merged.dropna()
df_wui_geneexp = df_wui_geneexp[(df_wui_geneexp.assay=='Immune') & df_wui_geneexp.anySigni]
# Calculate -log10(p-value)

ct_to_corrs = {}
for ct in celltypes:
    df_cut = df_wui_geneexp[df_wui_geneexp['assay']==ct]
    corr, p = spearmanr(df_cut['logFC'], df_cut['dWUI'])
    ct_to_corrs[ct] = corr

g = sns.relplot(
    data=df_wui_geneexp,
    x='dWUI',
    y='logFC',
    col='assay',
    kind='scatter',
    facet_kws={'sharey': False, 'sharex': False},
    hue='sig_direction',
    palette={'Shorter & signi': 'red', 'Longer & signi': 'blue', 'Other': 'gray'},
    col_wrap=1,  # Adjust based on number of assays
    height=3, 
    aspect=1,
    rasterized=True,
    zorder=10,
    s=10, alpha=0.9
)

g.set_axis_labels("ΔWUI", 'logFC')
g.set_titles("{col_name}")

for ax in g.axes.flat:
    ct = ax.get_title()
    ax.text(0.2, 0.99, f'r={ct_to_corrs[ct]:.2f}', ha='center', va='top', transform=ax.transAxes, fontsize=10)

for ax in g.axes.flat:
    ct = ax.get_title()
    ax.grid()
    
    df_sub = df_wui_geneexp[df_wui_geneexp['assay']==ct]
    texts = []
    for _, row in df_sub.iterrows():
        texts.append(ax.text(row['dWUI'], row['logFC'], row['gene_name'], clip_on=True, fontsize=8))
    adjust_text(texts, ax=ax, 
                avoid_self=True,
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

plt.savefig('logFC_dWUI_Immune_' + Path(du_path).name.replace('tsv.gz', 'pdf'), bbox_inches='tight', dpi=300)
plt.show()

## Violin

In [ ]:
df_wui_geneexp = df_merged.dropna()
sig_direction_order = ['Shorter & signi', 'Longer & signi', 'Other']

# Use catplot for violin
g = sns.catplot(
    data=df_wui_geneexp,
    x='sig_direction',
    hue='sig_direction',
    order=sig_direction_order,
    hue_order=sig_direction_order,
    y='logFC',
    col='assay',
    kind='violin',
    col_wrap=4,
    height=3,
    aspect=0.75,
    sharex=False, sharey=False
)

g.set_axis_labels("Significance Direction", "logFC")
g.set_titles("{col_name}")

# Add y-axis gridlines
for n, ax in enumerate(g.axes.flat):
    ax.yaxis.grid(True)
    ax.set_ylim([-.1, .1])
    if n>=4:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, va='top', ha='right', rotation_mode='anchor')
    else:
        ax.set_xticklabels([])

plt.show()

# Save as a table

In [ ]:
df_merged.to_csv('dWUI_and_geneExpr_' + Path(du_path).name, sep='\t', index=False)

In [ ]:
df_merged